In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.io import loadmat

from cqpsolver import Problem, Solver, SolverState

In [8]:
mat_dict: dict[np.ndarray] = loadmat("../QP-Test-Problems/MAT_Files/EXDATA.mat")

Q: sp.csc_array = sp.csc_array(mat_dict["Q"].astype(float))
q: np.ndarray = mat_dict["c"].astype(float)
A: sp.csc_array = sp.csc_array(mat_dict["A"].astype(float))
rl: np.ndarray = mat_dict["rl"].astype(float).flatten()
ru: np.ndarray = mat_dict["ru"].astype(float).flatten()
lb: np.ndarray = mat_dict["lb"].astype(float).flatten().reshape(-1, 1)
ub: np.ndarray = mat_dict["ub"].astype(float).flatten().reshape(-1, 1)

In [9]:
eq_mask: np.ndarray = rl == ru
A_eq: sp.csc_array = sp.csc_array(A[eq_mask])
b_eq: np.ndarray = ru[eq_mask].reshape(-1, 1)

A_eq: sp.csc_array = A_eq if A_eq.size > 0 else sp.csc_array((0, A.shape[1]))
b_eq: np.ndarray = b_eq if b_eq.size > 0 else np.zeros((0, 1))

ineq_mask: np.ndarray = np.invert(eq_mask)
G_ineq: sp.csc_array = sp.vstack([A[ineq_mask], -A[ineq_mask]], format="csc")
h_ineq: np.ndarray = np.concatenate([ru[ineq_mask], -rl[ineq_mask]]).reshape(-1, 1)

n: int = Q.shape[0]
G_full: sp.csc_array = sp.vstack([G_ineq, sp.eye(n), -sp.eye(n)], format="csc")
h_full: np.ndarray = np.vstack([h_ineq, ub, -lb])

finite_mask: np.ndarray = np.isfinite(h_full).flatten()
G: sp.csc_array = sp.csc_array(G_full[finite_mask])
h: np.ndarray = (h_full[finite_mask]).reshape(-1, 1)

In [10]:
prob: Problem = Problem(Q, q, G, h, A_eq, b_eq)
solver: Solver = Solver(prob, max_iter=100, quiet=False)
state_history: list[SolverState] = solver.solve()

─────────────────────────────────────────────────────────────────────────────────────────────────────
Iter. │   Objective    │ Primal Inequality │ Primal Equality │ Stationarity │   Duality   │ Step Size
─────────────────────────────────────────────────────────────────────────────────────────────────────
  0   │   -28.366678   │    1.1683e+02     │   5.2736e-15    │  1.3148e+03  │ 1.1197e+05  │     —    
  1   │   158.41314    │    5.1877e+00     │   7.3552e-16    │  5.8382e+01  │ 5.4571e+03  │  0.9556  
  2   │   119.07233    │    1.3199e+00     │   8.4377e-15    │  1.4855e+01  │ 1.5911e+03  │  0.7456  
  3   │    84.43383    │    7.2988e-01     │   1.1102e-16    │  8.2141e+00  │ 1.0941e+03  │  0.4470  
  4   │   45.654546    │    4.6958e-01     │   8.8263e-15    │  5.2847e+00  │ 7.9242e+02  │  0.3566  
  5   │   -1.9154185   │    2.7846e-01     │   9.1593e-15    │  3.1338e+00  │ 5.1935e+02  │  0.4070  
  6   │   -27.771301   │    1.9413e-01     │   6.4282e-14    │  2.1847e+00  │ 4.03